# Temporal and operational post-analysis — Infiltration, original labels

Run in Colab **on CPU**, in order. Reuses predictions saved by the ablation analysis under `results_stgnn_spatial_ablation_v1/diagnostic_test2_spatial`. If that cache is missing, first run the three prediction and comparison cells that accompanied the ablation notebook. This notebook **does not train**. It runs inference once for any missing original baselines and saves their probabilities to Drive.

The **main** tables compare ST-GNN against Simple MLP, E-GraphSAGE, Static GNN, and Edge GRU using validation-selected champions. The **ablation** tables compare ST-GNN with ST Uniform and ST No Message.

Graphs are 30-second windows indexed by flow start time. Detection delay is measured retrospectively from the first attack-labeled flow. Graph attributes contain statistics of completed flows, so this delay is not an online alert latency. MITRE stages are not inferred.


## 1. Setup and provenance

In [ ]:
import gc, hashlib, importlib.util, json, os, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score, f1_score

if importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/nids-mitre')
CODE_ROOT = Path('/content/temporalgnn-nids-st-ablation')
SOURCE_BRANCH = 'feat/fair-comparison-experiments'
SOURCE_COMMIT = '2262b53d155f9e8cf5221516f47e4170e6cd0d6e'
if not CODE_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_BRANCH,
                    'https://github.com/tatipar/temporalgnn-nids.git',
                    str(CODE_ROOT)], check=True)
actual_commit = subprocess.check_output(
    ['git', '-C', str(CODE_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_commit == SOURCE_COMMIT, f'Code commit differs: {actual_commit}'
sys.path.insert(0, str(CODE_ROOT / 'code/python'))

from utils.datasets import NF_IDS_Dataset
from utils.models import EdgeGRU_Baseline_NoX, SimpleMLP, E_GraphSAGE, StaticGNN_Identity
from utils.metrics import calculate_metrics_gnn

DEVICE = torch.device('cpu')
GRAPH_ROOT = DRIVE_ROOT / 'dataset_processed_thu0103'
ORIGINAL_RESULTS = DRIVE_ROOT / 'results_earlystopping'
ABLATION_RESULTS = DRIVE_ROOT / 'results_stgnn_spatial_ablation_v1'
PRED_DIR = ABLATION_RESULTS / 'diagnostic_test2_spatial'
POST_DIR = DRIVE_ROOT / 'results_comparison' / 'temporal_operational_v1'
POST_DIR.mkdir(parents=True, exist_ok=True)
assert PRED_DIR.is_dir(), f'Missing saved predictions: {PRED_DIR}'

dataset = NF_IDS_Dataset(root_dir=str(GRAPH_ROOT), split='test2')
assert len(dataset) == 2822, f'Expected 2,822 windows; found: {len(dataset)}'
assert [int(p.split('_')[1].split('.')[0]) for p in dataset.files] == list(range(2822)), (
    'Window IDs are not consecutive starting at zero'
)
GRAPH_START = pd.Timestamp(dataset[0].timestamp)
print('CPU:', __import__('platform').processor() or 'see /proc/cpuinfo')
print('Code:', SOURCE_COMMIT)
print('Start of first window:', GRAPH_START)
print('Results:', POST_DIR)


## 2. Saved predictions and window alignment

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


VARIANT_CACHES = {
    'ST-GNN': ('st_gnn_seed_42.npz', 42, 0.8384, 0.7576),
    'ST Uniform': ('st_uniform_seed_42.npz', 42, 0.6883, 0.6256),
    'ST No Message': ('st_no_message_seed_2024.npz', 2024, 0.6601, 0.7115),
}
predictions = {}
for name, (filename, seed, expected_ap, expected_f1) in VARIANT_CACHES.items():
    path = PRED_DIR / filename
    assert path.is_file(), f'Missing {path}; run the ablation analysis first'
    with np.load(path, allow_pickle=False) as saved:
        labels = saved['labels'].astype(np.int8, copy=False)
        sources = saved['sources'].astype(np.int64, copy=False)
        probs = saved['probabilities'].astype(np.float32, copy=False)
        threshold = float(saved['threshold'])
        cached_sha256 = str(saved['checkpoint_sha256'].item())
    if name == 'ST-GNN':
        original_exp = 'ST_GNN_BiasOn_robust_Identity_clone'
        original_csv = ORIGINAL_RESULTS / 'test2_results' / f'test2_metrics_{original_exp}.csv'
        original_row = pd.read_csv(original_csv).iloc[0]
        assert int(original_row['seed']) == seed
        original_run_id = str(original_row['run_id'])
        matches = list((ORIGINAL_RESULTS / 'saved_models' / original_exp).glob(f'{original_run_id}_*.pth'))
        assert len(matches) == 1, matches
        assert cached_sha256 == sha256_file(matches[0])
        with np.load(ORIGINAL_RESULTS / 'logs' / original_exp / f'thresholds_{original_exp}.npz') as saved_thresholds:
            assert threshold == float(saved_thresholds[f'seed_{seed}'])
    else:
        stem = name.lower().replace(' ', '_')
        record_path = ABLATION_RESULTS / stem / f'seed_{seed}' / 'record.json'
        checkpoint_path = record_path.parent / 'checkpoint.pth'
        record = json.loads(record_path.read_text())
        assert record['source_commit'] == SOURCE_COMMIT
        assert record['checkpoint_sha256'] == cached_sha256 == sha256_file(checkpoint_path)
        assert float(record['threshold']) == threshold
    metrics = calculate_metrics_gnn(labels, probs, prob_threshold=threshold)
    assert len(labels) == 2_135_957 and int(labels.sum()) == 103_825
    assert np.isclose(metrics['AUC-PR'], expected_ap, atol=0.0002), (name, metrics['AUC-PR'])
    assert np.isclose(metrics['F1'], expected_f1, atol=0.0002), (name, metrics['F1'])
    predictions[name] = dict(labels=labels, sources=sources, probs=probs,
                             threshold=threshold, seed=seed)
    print(f'{name}: AP={metrics["AUC-PR"]:.4f}; F1={metrics["F1"]:.4f}; cache verified')

reference = predictions['ST-GNN']
for name, result in predictions.items():
    np.testing.assert_array_equal(result['labels'], reference['labels'])
    np.testing.assert_array_equal(result['sources'], reference['sources'])

# Read graphs only; do not rerun inference for the three preceding models.
window_cache = POST_DIR / 'window_ids_test2.npz'
label_digest = hashlib.sha256(reference['labels'].tobytes()).hexdigest()
source_digest = hashlib.sha256(reference['sources'].tobytes()).hexdigest()
if window_cache.exists():
    with np.load(window_cache, allow_pickle=False) as saved:
        assert str(saved['label_sha256'].item()) == label_digest
        assert str(saved['source_sha256'].item()) == source_digest
        windows = saved['windows'].astype(np.int32, copy=False)
    print('Window IDs loaded from Drive')
else:
    w_parts, y_parts, s_parts = [], [], []
    for window_id in range(len(dataset)):
        graph = dataset[window_id]
        if graph.x.shape[0] == 0:
            continue
        count = int(graph.y.numel())
        w_parts.append(np.full(count, window_id, dtype=np.int32))
        y_parts.append(graph.y.numpy().astype(np.int8))
        s_parts.append(graph.global_node_ids[graph.edge_index[0]].numpy().astype(np.int64))
        if (window_id + 1) % 500 == 0:
            print('Graphs read:', window_id + 1)
    windows = np.concatenate(w_parts)
    np.testing.assert_array_equal(np.concatenate(y_parts), reference['labels'])
    np.testing.assert_array_equal(np.concatenate(s_parts), reference['sources'])
    tmp_path = POST_DIR / 'window_ids_test2.tmp.npz'
    np.savez_compressed(tmp_path, windows=windows,
                        label_sha256=np.array(label_digest),
                        source_sha256=np.array(source_digest))
    os.replace(tmp_path, window_cache)
assert len(windows) == len(reference['labels'])
print('Windows aligned with flows:', len(windows))


## 3. Original baselines: CPU inference without retraining

In [ ]:
EDGE_EXP = 'EdgeGRU_NoX_BiasOn'
edge_csv = ORIGINAL_RESULTS / 'test2_results' / f'test2_metrics_{EDGE_EXP}.csv'
edge_row = pd.read_csv(edge_csv).iloc[0]
edge_seed = int(edge_row['seed'])
edge_run_id = str(edge_row['run_id'])
assert edge_seed == 99
edge_matches = list((ORIGINAL_RESULTS / 'saved_models' / EDGE_EXP).glob(f'{edge_run_id}_*.pth'))
assert len(edge_matches) == 1, edge_matches
edge_checkpoint = edge_matches[0]
edge_hash = sha256_file(edge_checkpoint)
with np.load(ORIGINAL_RESULTS / 'logs' / EDGE_EXP / f'thresholds_{EDGE_EXP}.npz') as saved:
    edge_threshold = float(saved[f'seed_{edge_seed}'])

edge_cache = POST_DIR / 'edge_gru_seed_99_test2.npz'
if edge_cache.exists():
    with np.load(edge_cache, allow_pickle=False) as saved:
        assert str(saved['checkpoint_sha256'].item()) == edge_hash
        assert float(saved['threshold']) == edge_threshold
        edge_labels = saved['labels'].astype(np.int8, copy=False)
        edge_sources = saved['sources'].astype(np.int64, copy=False)
        edge_probs = saved['probabilities'].astype(np.float32, copy=False)
    print('Edge GRU: predictions loaded from Drive')
else:
    model = EdgeGRU_Baseline_NoX(node_dim=16, edge_dim=32, hidden_dim=32,
                                 dropout=0.2, output_bias_init=-2.9968).to(DEVICE)
    state = torch.load(edge_checkpoint, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state, strict=True)
    model.eval()
    model.reset_memory()
    y_parts, s_parts, p_parts = [], [], []
    with torch.no_grad():
        for window_id in range(len(dataset)):
            graph = dataset[window_id]
            if graph.x.shape[0] == 0:
                continue
            logits = model(graph.x, graph.edge_index, graph.edge_attr,
                           graph.global_node_ids).view(-1)
            y_parts.append(graph.y.numpy().astype(np.int8))
            s_parts.append(graph.global_node_ids[graph.edge_index[0]].numpy().astype(np.int64))
            p_parts.append(torch.sigmoid(logits).numpy().astype(np.float32))
            if (window_id + 1) % 500 == 0:
                print('Edge GRU: windows processed', window_id + 1)
    edge_labels = np.concatenate(y_parts)
    edge_sources = np.concatenate(s_parts)
    edge_probs = np.concatenate(p_parts)
    tmp_path = POST_DIR / 'edge_gru_seed_99_test2.tmp.npz'
    np.savez_compressed(tmp_path, labels=edge_labels, sources=edge_sources,
                        probabilities=edge_probs, threshold=np.array(edge_threshold),
                        checkpoint_sha256=np.array(edge_hash))
    os.replace(tmp_path, edge_cache)
    del model, state
    gc.collect()
    print('Edge GRU: predictions saved to Drive')

np.testing.assert_array_equal(edge_labels, reference['labels'])
np.testing.assert_array_equal(edge_sources, reference['sources'])
edge_metrics = calculate_metrics_gnn(edge_labels, edge_probs, prob_threshold=edge_threshold)
assert np.isclose(edge_metrics['AUC-PR'], 0.7307, atol=0.0002), edge_metrics['AUC-PR']
assert np.isclose(edge_metrics['F1'], 0.6502, atol=0.0002), edge_metrics['F1']
predictions['Edge GRU'] = dict(labels=edge_labels, sources=edge_sources, probs=edge_probs,
                               threshold=edge_threshold, seed=edge_seed)
print(f'Edge GRU reproduced: AP={edge_metrics["AUC-PR"]:.4f}; F1={edge_metrics["F1"]:.4f}')

BASE_PARAMS = dict(node_dim=16, edge_dim=32, hidden_dim=32,
                   dropout=0.2, output_bias_init=-2.9968)
BASELINE_SPECS = [
    ('Simple MLP', 'SimpleMLP_BiasOn', SimpleMLP, 777, 0.3961, 0.4722),
    ('E-GraphSAGE', 'EGraphSAGE_BiasOn', E_GraphSAGE, 99, 0.5050, 0.4950),
    ('Static GNN', 'StaticGNN_BiasOn_robust_Identity', StaticGNN_Identity, 42, 0.3142, 0.2944),
]
for name, experiment, model_class, expected_seed, expected_ap, expected_f1 in BASELINE_SPECS:
    reported_csv = ORIGINAL_RESULTS / 'test2_results' / f'test2_metrics_{experiment}.csv'
    reported = pd.read_csv(reported_csv).iloc[0]
    seed = int(reported['seed'])
    assert seed == expected_seed, (name, seed)
    run_id = str(reported['run_id'])
    matches = list((ORIGINAL_RESULTS / 'saved_models' / experiment).glob(f'{run_id}_*.pth'))
    assert len(matches) == 1, (name, matches)
    checkpoint = matches[0]
    checkpoint_hash = sha256_file(checkpoint)
    with np.load(ORIGINAL_RESULTS / 'logs' / experiment / f'thresholds_{experiment}.npz') as saved:
        threshold = float(saved[f'seed_{seed}'])

    cache = POST_DIR / f'{experiment}_seed_{seed}_test2.npz'
    if cache.exists():
        with np.load(cache, allow_pickle=False) as saved:
            assert str(saved['checkpoint_sha256'].item()) == checkpoint_hash
            assert float(saved['threshold']) == threshold
            labels_i = saved['labels'].astype(np.int8, copy=False)
            sources_i = saved['sources'].astype(np.int64, copy=False)
            probs_i = saved['probabilities'].astype(np.float32, copy=False)
        print(f'{name}: predictions loaded from Drive')
    else:
        model = model_class(**BASE_PARAMS).to(DEVICE)
        state = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
        model.load_state_dict(state, strict=True)
        model.eval()
        y_parts, s_parts, p_parts = [], [], []
        with torch.no_grad():
            for window_id in range(len(dataset)):
                graph = dataset[window_id]
                if graph.x.shape[0] == 0:
                    continue
                logits = model(graph.x, graph.edge_index, graph.edge_attr).view(-1)
                y_parts.append(graph.y.numpy().astype(np.int8))
                s_parts.append(graph.global_node_ids[graph.edge_index[0]].numpy().astype(np.int64))
                p_parts.append(torch.sigmoid(logits).numpy().astype(np.float32))
                if (window_id + 1) % 500 == 0:
                    print(f'{name}: windows processed', window_id + 1)
        labels_i = np.concatenate(y_parts)
        sources_i = np.concatenate(s_parts)
        probs_i = np.concatenate(p_parts)
        tmp_path = POST_DIR / f'{experiment}_seed_{seed}_test2.tmp.npz'
        np.savez_compressed(tmp_path, labels=labels_i, sources=sources_i,
                            probabilities=probs_i, threshold=np.array(threshold),
                            checkpoint_sha256=np.array(checkpoint_hash))
        os.replace(tmp_path, cache)
        del model, state
        gc.collect()
        print(f'{name}: predictions saved to Drive')

    np.testing.assert_array_equal(labels_i, reference['labels'])
    np.testing.assert_array_equal(sources_i, reference['sources'])
    metrics_i = calculate_metrics_gnn(labels_i, probs_i, prob_threshold=threshold)
    assert np.isclose(metrics_i['AUC-PR'], expected_ap, atol=0.0002), (name, metrics_i['AUC-PR'])
    assert np.isclose(metrics_i['F1'], expected_f1, atol=0.0002), (name, metrics_i['F1'])
    predictions[name] = dict(labels=labels_i, sources=sources_i, probs=probs_i,
                             threshold=threshold, seed=seed)
    print(f'{name} reproduced: AP={metrics_i["AUC-PR"]:.4f}; F1={metrics_i["F1"]:.4f}')

PRIMARY_MODELS = ['Simple MLP', 'E-GraphSAGE', 'Static GNN', 'Edge GRU', 'ST-GNN']
ABLATION_MODELS = ['ST-GNN', 'ST Uniform', 'ST No Message']
assert set(predictions) == set(PRIMARY_MODELS) | set(ABLATION_MODELS)

metric_rows = []
for name, result in predictions.items():
    scores = calculate_metrics_gnn(result['labels'], result['probs'],
                                   prob_threshold=result['threshold'])
    metric_rows.append({'model': name, 'seed': result['seed'],
                        'AP': scores['AUC-PR'], 'F1': scores['F1'],
                        'precision': scores['Precision'], 'recall': scores['Recall'],
                        'FPR': scores['FPR'], 'TP': scores['TP'], 'FP': scores['FP']})
metric_table = pd.DataFrame(metric_rows)
for title, model_names, suffix in [
    ('Main comparison: flow metrics, Test 2', PRIMARY_MODELS, 'baselines'),
    ('Ablations: flow metrics, Test 2', ABLATION_MODELS, 'ablations'),
]:
    print(title)
    selected = metric_table.set_index('model').loc[model_names].reset_index()
    selected.to_csv(POST_DIR / f'flow_metrics_{suffix}.csv', index=False)
    display(selected.round(4))


## 4. First detection and continuity

The analysis unit is an **attacked source in Test 2**. A hit is an alert on an attack-labeled flow. Delay is measured from that source's first window with an attack label. Sources never receiving a true positive have censored delay and are counted separately. Incidents and attack stages are not reconstructed.


In [ ]:
labels = reference['labels']
sources = reference['sources']
attacked_sources = np.unique(sources[labels == 1])


def longest_consecutive_missed(active_windows, hit_windows):
    hits = set(int(w) for w in hit_windows)
    longest = current = 0
    previous = None
    for raw_window in active_windows:
        window = int(raw_window)
        if window in hits:
            current = 0
        else:
            current = current + 1 if previous is not None and window == previous + 1 else 1
            longest = max(longest, current)
        previous = window
    return longest


source_rows = []
for name, result in predictions.items():
    alert = result['probs'] > result['threshold']  # original decision operator
    attack = labels == 1
    relevant = pd.DataFrame({
        'source': sources[attack],
        'window': windows[attack],
        'hit': alert[attack],
    })
    for source, group in relevant.groupby('source', sort=True):
        active = np.sort(group['window'].unique())
        hits = np.sort(group.loc[group['hit'], 'window'].unique())
        first_hit = int(hits[0]) if len(hits) else None
        delay = (first_hit - int(active[0])) if first_hit is not None else None
        before_first = int((active < first_hit).sum()) if first_hit is not None else len(active)
        after_first = int((active >= first_hit).sum()) if first_hit is not None else 0
        source_rows.append({
            'model': name,
            'source_gid': int(source),
            'attack_flows': len(group),
            'attack_windows': len(active),
            'first_attack_window': int(active[0]),
            'first_hit_window': first_hit,
            'detected': first_hit is not None,
            'delay_windows': delay,
            'nominal_delay_minutes': (delay * 0.5 if delay is not None else np.nan),
            'attack_windows_detected': len(hits),
            'attack_window_coverage': len(hits) / len(active),
            'attack_windows_before_first_hit': before_first,
            'coverage_after_first_hit': len(hits) / after_first if after_first else np.nan,
            'longest_consecutive_missed_windows': longest_consecutive_missed(active, hits),
        })

source_table = pd.DataFrame(source_rows)
assert source_table.groupby('model').size().eq(len(attacked_sources)).all()
source_table.to_csv(POST_DIR / 'source_detection.csv', index=False)

summary_rows = []
for subset_name, included in [
    ('All attacked sources', source_table['source_gid'] >= 0),
    ('Excluding source 1195', source_table['source_gid'] != 1195),
]:
    for name, group in source_table[included].groupby('model'):
        detected = group['detected']
        delay = group['nominal_delay_minutes']
        summary_rows.append({
            'subset': subset_name,
            'model': name,
            'attacked_sources': len(group),
            'sources_detected': int(detected.sum()),
            'sources_never_detected': int((~detected).sum()),
            'median_minutes_detected_only': delay.median(),
            'detected_in_first_window': int((group['delay_windows'] == 0).sum()),
            'detected_within_5_min': int((delay <= 5).sum()),
            'detected_within_30_min': int((delay <= 30).sum()),
            'mean_source_coverage': group['attack_window_coverage'].mean(),
            'window_weighted_coverage': group['attack_windows_detected'].sum() / group['attack_windows'].sum(),
            'median_longest_miss_streak': group['longest_consecutive_missed_windows'].median(),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(POST_DIR / 'source_temporal_summary.csv', index=False)
for title, model_names, suffix in [
    ('Main comparison: ST-GNN and baselines', PRIMARY_MODELS, 'baselines'),
    ('Ablations: ST-GNN and variants', ABLATION_MODELS, 'ablations'),
]:
    print(title)
    selected = summary[summary.model.isin(model_names)]
    selected.to_csv(POST_DIR / f'temporal_summary_{suffix}.csv', index=False)
    display(selected.round(3))
    print('Source 1195:')
    display(source_table[(source_table.source_gid == 1195) &
                         source_table.model.isin(model_names)].round(3))

# Median delay above is conditional on source detection.
# This breakdown checks whether source coverage depends on sources with
# very few attack-labeled flows.
source_table['attack_count_group'] = pd.cut(
    source_table['attack_flows'], bins=[0, 1, 9, 49, np.inf],
    labels=['1', '2-9', '10-49', '50+']
)
support_rows = []
for (name, group_name), group in source_table.groupby(['model', 'attack_count_group'], observed=True):
    support_rows.append({
        'model': name,
        'attack_flows_per_source': str(group_name),
        'sources': len(group),
        'detected': int(group['detected'].sum()),
        'fraction_detected': group['detected'].mean(),
        'mean_window_coverage': group['attack_window_coverage'].mean(),
    })
support_table = pd.DataFrame(support_rows)
support_table.to_csv(POST_DIR / 'source_support_detection.csv', index=False)
for title, model_names, suffix in [
    ('Baselines: coverage by source attack count', PRIMARY_MODELS, 'baselines'),
    ('Ablations: coverage by source attack count', ABLATION_MODELS, 'ablations'),
]:
    print(title)
    selected = support_table[support_table.model.isin(model_names)]
    selected.to_csv(POST_DIR / f'support_detection_{suffix}.csv', index=False)
    display(selected.round(3))


## 5. CPU alert workload: flows and source-window pairs

Each model keeps its validation-selected threshold. A source-window pair containing any attack-labeled flow is an attack window. The analysis separates alerts with a true positive, alerts with no labeled attack, and alerts on benign flows in an attack window that contain no true positive. Counts depend on the original labels.


In [ ]:
base_pairs = pd.DataFrame({
    'window': windows,
    'source': sources,
    'attack': labels == 1,
})
attack_pairs = base_pairs.loc[base_pairs.attack, ['window', 'source']].drop_duplicates()
attack_pairs['window_contains_attack'] = True
N_HOURS = (len(dataset) + 119) // 120  # 120 thirty-second windows per hour

workload_rows = []
hourly_rows = []
for name, result in predictions.items():
    alert = result['probs'] > result['threshold']
    alert_df = pd.DataFrame({
        'window': windows[alert],
        'source': sources[alert],
        'attack': labels[alert] == 1,
    })
    groups = alert_df.groupby(['window', 'source'], sort=False).agg(
        alerted_flows=('attack', 'size'),
        has_true_positive=('attack', 'any'),
    ).reset_index()
    groups = groups.merge(attack_pairs, on=['window', 'source'], how='left', validate='one_to_one')
    groups['window_contains_attack'] = groups['window_contains_attack'].eq(True)
    groups['false_source_window'] = ~groups['window_contains_attack']
    groups['only_benign_alert_in_attack_window'] = (
        groups['window_contains_attack'] & ~groups['has_true_positive']
    )
    workload_rows.append({
        'model': name,
        'alerted_flows': int(alert.sum()),
        'FP_flows': int((alert & (labels == 0)).sum()),
        'alerted_source_window_pairs': len(groups),
        'pairs_with_true_positive': int(groups['has_true_positive'].sum()),
        'pairs_without_labeled_attack': int(groups['false_source_window'].sum()),
        'attack_pairs_with_only_benign_alerts': int(groups['only_benign_alert_in_attack_window'].sum()),
    })

    flow_hours = pd.DataFrame({
        'hour': windows[alert] // 120,
        'FP': labels[alert] == 0,
    }).groupby('hour').agg(flow_alerts=('FP', 'size'), FP_flows=('FP', 'sum'))
    group_hours = groups.assign(hour=groups['window'] // 120).groupby('hour').agg(
        source_window_alerts=('source', 'size'),
        source_window_alerts_without_attack=('false_source_window', 'sum'),
    )
    hours = flow_hours.join(group_hours, how='outer').reindex(range(N_HOURS), fill_value=0).fillna(0)
    for hour, row in hours.iterrows():
        hourly_rows.append({'model': name, 'hour_since_start': int(hour),
                            **{column: int(value) for column, value in row.items()}})

workload = pd.DataFrame(workload_rows)
hourly = pd.DataFrame(hourly_rows)
workload.to_csv(POST_DIR / 'alert_workload.csv', index=False)
hourly.to_csv(POST_DIR / 'hourly_alert_workload.csv', index=False)
def p95(values):
    return values.quantile(0.95)

for title, model_names, suffix in [
    ('Main comparison: ST-GNN and baselines', PRIMARY_MODELS, 'baselines'),
    ('Ablations: ST-GNN and variants', ABLATION_MODELS, 'ablations'),
]:
    print(title)
    selected_workload = workload[workload.model.isin(model_names)]
    selected_hourly = hourly[hourly.model.isin(model_names)]
    selected_workload.to_csv(POST_DIR / f'alert_workload_{suffix}.csv', index=False)
    selected_hourly.to_csv(POST_DIR / f'hourly_alert_workload_{suffix}.csv', index=False)
    display(selected_workload)
    display(selected_hourly.groupby('model')[[
        'flow_alerts', 'FP_flows', 'source_window_alerts',
        'source_window_alerts_without_attack'
    ]].agg(['median', p95, 'max']).round(1))
print('Medians may be zero when many hours have no alerts; inspect p95 and maximum as well.')
print('The final hour may be partial; the hourly table retains each interval.')


## 6. Availability audit for completed-flow attributes

This audit reads the original CSV in chunks and counts flows whose duration exceeds the time remaining until the end of their start window. This **suggests** that their complete statistics would not be available at that window boundary. It does not estimate exporter delay or turn this experiment into an online deployment.


In [ ]:
RAW_CSV = DRIVE_ROOT / 'data/cicids2018-v3/cicids2018v3_thu0103.csv'
if not RAW_CSV.is_file():
    print('Original CSV not found; skipping this audit:', RAW_CSV)
else:
    columns = ['FLOW_START_TIME', 'FLOW_DURATION_MILLISECONDS',
               'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Attack']
    graph_rows = valid_duration_rows = crossed_rows = attack_rows = crossed_attack_rows = 0
    invalid_duration_rows = 0
    all_durations = []
    for chunk_no, chunk in enumerate(pd.read_csv(RAW_CSV, usecols=columns, chunksize=250_000), start=1):
        valid_ip = ((chunk['IPV4_SRC_ADDR'] != '0.0.0.0') &
                    (chunk['IPV4_DST_ADDR'] != '0.0.0.0'))
        chunk = chunk.loc[valid_ip]
        graph_rows += len(chunk)
        starts = pd.to_datetime(chunk['FLOW_START_TIME'], errors='coerce')
        durations = pd.to_numeric(chunk['FLOW_DURATION_MILLISECONDS'], errors='coerce')
        valid = starts.notna() & np.isfinite(durations) & (durations >= 0)
        invalid_duration_rows += int((~valid).sum())
        starts, durations = starts[valid], durations[valid]
        attacks = chunk.loc[valid, 'Attack'].eq('Infilteration')
        valid_duration_rows += len(starts)
        attack_rows += int(attacks.sum())
        offset_ms = (starts - GRAPH_START).dt.total_seconds().to_numpy() * 1000.0
        assert np.all(offset_ms >= 0), 'Some flows precede the start of the graphs'
        remaining_ms = 30_000.0 - np.mod(offset_ms, 30_000.0)
        cross = durations.to_numpy() > remaining_ms
        crossed_rows += int(cross.sum())
        crossed_attack_rows += int((cross & attacks.to_numpy()).sum())
        all_durations.append(durations.to_numpy(dtype=np.float64))
        print(f'Chunk {chunk_no}: {len(chunk)} flows with valid IPs')

    assert graph_rows == len(labels), (graph_rows, len(labels))
    duration_array = np.concatenate(all_durations)
    timing_audit = pd.DataFrame([{
        'flows_in_graphs': graph_rows,
        'flows_with_valid_duration': valid_duration_rows,
        'invalid_duration_count': invalid_duration_rows,
        'median_duration_ms': float(np.median(duration_array)),
        'p95_duration_ms': float(np.percentile(duration_array, 95)),
        'cross_start_window_end': crossed_rows,
        'fraction_cross_start_window_end': crossed_rows / valid_duration_rows,
        'attacks_with_valid_duration': attack_rows,
        'attacks_cross_start_window_end': crossed_attack_rows,
        'fraction_attacks_cross': crossed_attack_rows / attack_rows if attack_rows else np.nan,
    }])
    timing_audit.to_csv(POST_DIR / 'flow_availability_audit.csv', index=False)
    display(timing_audit.round(4))


## Interpretation and scope

- Baseline tables are the paper's main comparison. Ablation tables describe changes to ST-GNN; by themselves they neither replace baselines nor establish causality.
- Nominal delay in minutes equals the window difference times 0.5. It refers to graphs indexed by flow start time. Never-detected sources remain in `sources_never_detected`.
- `window_weighted_coverage` counts each attack window for each source; `mean_source_coverage` gives each source equal weight.
- `pairs_without_labeled_attack` measures alerts grouped by source and window using original labels. Mixed groups with alerts only on benign flows are reported separately.
- Report the CPU model from the Colab output. Timing in the comparative notebook measures only `forward` calls on prebuilt graphs; this notebook does not present those times as end-to-end latency.
